In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import sqlite3

project_root = Path.cwd().parent
sys.path.append(str(project_root))

In [2]:
from infrastructure.load_db import DatabaseManager
from domain.embedder import Embedder
from infrastructure.sentence_transformer_embedder import SentenceTransformerEmbedder

/Users/patty/Documents/Greta courses/Course/Dev IA/Dev_IA_Cinedata/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1029 14:43:30.482000 79583 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
conn = DatabaseManager().load_db()
conn.row_factory = sqlite3.Row
embedder: Embedder = SentenceTransformerEmbedder()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7967.20it/s]


### Load chunk embeddings

In [4]:
rows = conn.execute("""
    SELECT e.movielens_id, e.chunk_index, e.vector
    FROM chunk_embeddings e
    WHERE e.model = ?
    ORDER BY e.movielens_id, e.chunk_index
""", (embedder.model_name,)).fetchall()

keys = [(r[0], r[1]) for r in rows]
matrix = np.vstack([np.frombuffer(r[2], dtype=np.float32) for r in rows])

In [5]:
question = "A toy cowboy gets jealous when the owner's new astronaut toy becomes his favourite"

q = embedder.embed([question])[0]
q.dtype

dtype('float32')

In [6]:
matrix.shape

(27138, 384)

In [7]:
scores = matrix @ q
scores.shape

(27138,)

### Sort by scores

In [8]:
limit = 10

best_rows = np.argsort(scores)[::-1][:limit] 

In [9]:
best_ids = [ keys[index] for index in best_rows ]
best_ids

[(1, 0),
 (3114, 0),
 (1, 3),
 (3114, 4),
 (4777, 0),
 (1, 2),
 (1, 1),
 (3827, 1),
 (50923, 1),
 (51698, 0)]

In [10]:
best_scores = [ scores[index] for index in best_rows ]
best_scores

[np.float32(0.56443185),
 np.float32(0.51275915),
 np.float32(0.5036428),
 np.float32(0.5035743),
 np.float32(0.501276),
 np.float32(0.49790114),
 np.float32(0.49067315),
 np.float32(0.47543025),
 np.float32(0.47309265),
 np.float32(0.47056776)]

### Search text by ids

In [11]:
SEARCH_TEXT = """
    SELECT m.title, m.year, pc.text
    FROM plot_chunks pc
    JOIN movies m ON m.movielens_id = pc.movielens_id
    WHERE pc.movielens_id = ?
    AND pc.chunk_index = ?
"""

In [12]:
search_rows = []

for movie_id, chunk_id in best_ids:
    row = conn.execute(
        SEARCH_TEXT,
        (movie_id, chunk_id)
    ).fetchone()
    search_rows.append(row)

for score, row in zip(best_scores, search_rows):
    print(f"Score: {score} for {row[0]} ({row[1]}), {row[2][:200]}")

Score: 0.5644318461418152 for Toy Story (1995), In a world where toys are sentient but pretend to be lifeless around humans, a group of toys, led by Sheriff Woody, are preparing to move to a new house with their young owner, Andy Davis, and his fam
Score: 0.5127591490745544 for Toy Story 2 (1999), Woody and Buzz Lightyear, having become best friends, are now the co-leaders of Andy's toys. Andy plans to take Woody to Cowboy Camp, but accidentally rips his arm during a brief playtime. Andy's moth
Score: 0.5036427974700928 for Toy Story (1995), After Sid flees, Woody and Buzz pursue the Davises' moving truck, but Sid's dog Scud attacks Woody. While Buzz fights off Scud, Woody climbs into the truck and pushes RC out to rescue Buzz. The other 
Score: 0.5035743117332458 for Toy Story 2 (1999), Andy's Buzz and the other toys pursue Al to the airport by driving a Pizza Planet truck, accompanied by three toy aliens from the restaurant. Buzz and the rest of Andy's toys pursue the baggage throug


## Evaluation

In [13]:
from infrastructure.sqlite_embedding_repository import SqliteEmbeddingRepository
from infrastructure.sqlite_chunk_repository import SqliteChunkRepository
from services.plot_search_service import PlotSearchService

embedder = SentenceTransformerEmbedder()
index = SqliteEmbeddingRepository(conn).load_chunk_index(embedder.model_name)
service = PlotSearchService(embedder, index)
chunk_repo = SqliteChunkRepository(conn)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10951.74it/s]


In [24]:
def show(question: str, limit: int = 5) -> None:
    hits = service.search(question, limit)
    passages = chunk_repo.get_passages([(m, c) for m, c, _ in hits])
    for (_, _, score), passage in zip(hits, passages):
        print(f"Score: {score}, {passage.movielens_id} {passage.title} ({passage.year}) {passage.text[:200]}")

In [31]:
show("The leader, Barbero, welcomes them warmly but locks them in an attic")

Score: 0.4031602740287781, 1281 Great Dictator, The (1940) Escaping from the camp in stolen uniforms, Schultz and the barber, dressed as Hynkel, arrive at the Osterlich frontier, where a victory parade crowd is waiting to be addressed by Hynkel. The real Hynk
Score: 0.4006270170211792, 1281 Great Dictator, The (1940) Twenty years later, still suffering from amnesia, the Jewish soldier returns to his previous profession as a barber in a ghetto. The ghetto is now governed by Schultz who has been promoted in the Toma
Score: 0.3879341781139374, 55721 Elite Squad (Tropa de Elite) (2007) In 1997, Captain Roberto Nascimento of the Battalion of Special Police Operations (BOPE) leads an operation to secure the Turano neighbourhood before Pope John Paul II's overnight visit at the Archbis
Score: 0.38217175006866455, 5971 My Neighbor Totoro (Tonari no Totoro) (1988) In 1950s Japan, university professor Tatsuo Kusakabe and his daughters Satsuki and Mei (approximately ten and four years old, respec

In [16]:
from domain.evaluation import is_hit

### Test on known films

In [61]:
EVAL = [
    ("Woody is jealous when Andy gets a new Buzz Lightyear toy", {1}),
    ("Three women in a small town are seduced by a mysterious wealthy stranger", {7046}),
    ("Romani street merchant who sells him an old DVD of The Man Who Killed Don", {188833}),
    ("A hacker is contacted by a mysterious group who tell him the world isn’t real", {2571}),
    ("Jack decides to bring christmas to the town", {551}),
    ("A man with scissor as hands", {2291}),
    ("A couple who investigates ghosts. the most famous one is called Annabelle", {103688}),
    ("A woman arrives at a gay conversion camp but falls in love with a girl", {3786}),
    ("A man in wheelchair watches other apartments from the window and witnesses a murder", {904}),
    ("Detectives search for a serial killer who follows seven sins", {47})
]

for text, expected_id in EVAL:
    hits = service.search(text)
    ids = [ m for m, _, _ in hits ]
    print(
        is_hit(
            retrieved_ids=ids,
            expected_ids=expected_id
        )
    )

True
False
True
True
True
True
True
True
False
True


In [58]:
def rank_of(question: str, expected_ids: set[int], depth: int = 100) -> int | None:
    hits = service.search(question, depth)
    for index in range(depth):
        m, c, s = hits[index]
        if m in expected_ids:
            print(f"FOUND! Index: {index}, {m} | {s}")
            return index
    
    print(f"Not found")
    show(question, 3)

    return None

In [59]:
for text, expected_id in EVAL:
    rank_of(
        question=text,
        expected_ids=expected_id,
        depth=30
    )

FOUND! Index: 0, 1 | 0.7078595161437988
Not found
Score: 0.46525177359580994, 8261 3 Women (Three Women) (1977) Millie becomes increasingly frustrated by Pinky's imitative shift in personality and begins to exhibit Pinky's timid and submissive personality herself. One night after Pinky has a bad dream, she shar
Score: 0.43222102522850037, 8261 3 Women (Three Women) (1977) Pinky Rose, a timid and awkward young woman, starts working at a health spa for the elderly in a small California desert town. She becomes enamored of Millie Lammoreaux, a relentlessly outgoing and se
Score: 0.4310966432094574, 8261 3 Women (Three Women) (1977) Later, Pinky and Millie are working at Dodge City, having again changed roles: Pinky has reverted to her childlike timidity and refers to Millie as her mother, while Millie has assumed Willie's role i
FOUND! Index: 0, 188833 | 0.47030699253082275
FOUND! Index: 1, 2571 | 0.4522993266582489
FOUND! Index: 0, 551 | 0.6187735795974731
FOUND! Index: 0, 2291 | 0.43354

### Check the missed ones

In [62]:
MISS_EVAL = [
    ("Three women in a small town are seduced by a mysterious wealthy stranger", {7046}),
    ("A man in wheelchair watches other apartments from the window and witnesses a murder", {904}),
]

for text, expected_id in MISS_EVAL:
    rank_of(
        question=text,
        expected_ids=expected_id,
        depth=100
    )

FOUND! Index: 80, 7046 | 0.3105790615081787
Not found
Score: 0.41881370544433594, 5556 FearDotCom (a.k.a. Fear.com) (a.k.a. Fear Dot Com) (2002) Mike Reilly is an NYPD detective who is called to the scene of a mysterious death in the subway system. The victim, Polidori, exhibits bleeding from his eyes and other orifices and, by the frozen look
Score: 0.417961061000824, 67695 Observe and Report (2009) An anonymous flasher exposes himself to shoppers in the Forest Ridge Mall parking lot. The head of mall security, Ronnie Barnhardt, makes it his mission to apprehend the offender. But while he is oste
Score: 0.4167144298553467, 3445 Eyes of Laura Mars (1978) At her apartment, Laura is contacted by Neville who tells her it's all over, advising her to pack a bag so they can go away together. Laura is then affected by a vision of the killer murdering Michael
